In [1]:
from llms import roberta, evaluator, roberta_legal_retriever
import polars as pl

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
roberta = roberta()
evaluator = evaluator()

Setup complete. Caching and Output mapped to SSD: {ssd_base_path}


ValueError: Either results_csv_path or results_df must be provided.

In [ ]:
cases_datasets_paths = [
    "/mnt/red/red_hanka_bcthesis/llm/finetuning_EStGdataset_1.csv",
    "/mnt/red/red_hanka_bcthesis/llm/finetuning_UStGdataset_1.csv",
    "/mnt/red/red_hanka_bcthesis/llm/finetuning_BAOdataset_1.csv"
]
# estg_cases_path = "/mnt/red/red_hanka_bcthesis/llm/finetuning_EStGdataset_1.csv"
# ustg_cases_path = "/mnt/red/red_hanka_bcthesis/llm/finetuning_UStGdataset_1.csv"
# bao_cases_path = "/mnt/red/red_hanka_bcthesis/llm/finetuning_BAOdataset_1.csv"
# Load data immediately
try:
    cases_sample, cited_cases_sample = roberta.load_datasets(cases_path = cases_datasets_paths)
except:
    print("Data loading failed. Please ensure 'training_cases.csv' and 'test_cases.csv' are present.")


In [ ]:
#finetune model
if __name__ == "__main__":
    # 1. Initialize the Finetuner
    roberta_bot = roberta()

    # 2. Load your datasets
    cases_datasets_paths = [
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_EStGdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_UStGdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_BAOdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_ASVGdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_DBAdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_GrEStGdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_KStGdataset_1.csv"
    ]
    train_df = roberta_bot.load_datasets(cases_datasets_paths)

    # # 🔥 THE SMOKE TEST HACK 🔥
    # # This chops your massive dataset down to just the first 10 rows for testing
    # train_df = train_df.head(10)
    # print(f"⚠️ SMOKE TEST MODE: Only training on {train_df.height} cases.")

    # 3. Load Base Model and Fine-Tune
    base_model = roberta_bot.load_model()
    finetuned_roberta = roberta_bot.finetune_model(base_model, train_df)

    # 4. Extract unique citations for the FAISS Hack
    print("\nExtracting unique citations for FAISS index...")
    # Polars magic to get a flat list of unique citations
    unique_cits = (
        train_df.select(pl.col("parsed_citations").list.explode())
        .drop_nulls()
        .unique()
        .to_series()
        .to_list()
    )

    # 5. Build the Retriever (The Librarian)
    retriever = roberta_legal_retriever(model=finetuned_roberta, unique_citations=unique_cits)

    # 6. Run a Quick Sanity Check Test
    print("\n" + "="*50)
    print("LIVE RETRIEVAL TEST")
    print("="*50)

    # Testen wir direkt das KStG (Körperschaftsteuer), um zu sehen, ob es gelernt hat!
    test_case = "Welche steuerlichen Konsequenzen hat es, wenn eine GmbH ein zinsloses Darlehen an ihren Gesellschafter vergibt?"
    print(f"QUERY: {test_case}\n")

    results = retriever.retrieve(test_case, k=5)
    for r in results:
        print(f"Rank {r['rank']}: {r['citation']} (Score: {r['score']:.4f})")

In [ ]:
#roberta eval 1st round
print("\n" + "="*50)
print("FINAL EVALUATION METRICS")
print("="*50)

# Instantiate your existing evaluator class
eval_engine = evaluator(results_csv_path=BASELINE_CSV)  # You can initialize with any CSV, it just needs the path for internal use

print("\n--- BASELINE SCORES ---")
eval_engine.evaluate_results(
    output_path="/mnt/red/red_hanka_bcthesis/llm/eval_roberta_baseline_1.csv", 
    results_csv_path=BASELINE_CSV
)

print("\n--- FINE-TUNED SCORES ---")
eval_engine.evaluate_results(
    output_path="/mnt/red/red_hanka_bcthesis/llm/eval_roberta_finetuned_1.csv", 
    results_csv_path=FINETUNED_CSV
)


FINAL EVALUATION METRICS

--- BASELINE SCORES ---
Loading results from /mnt/red/red_hanka_bcthesis/llm/roberta_baseline_results_1.csv...
shape: (200, 10)
┌────────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬────────┐
│ id         ┆ instructio ┆ input     ┆ ground_tr ┆ … ┆ pred_list ┆ exact_mat ┆ precision ┆ recall │
│ ---        ┆ n          ┆ ---       ┆ uth_label ┆   ┆ ---       ┆ ch        ┆ ---       ┆ ---    │
│ str        ┆ ---        ┆ str       ┆ ---       ┆   ┆ list[str] ┆ ---       ┆ f64       ┆ f64    │
│            ┆ str        ┆           ┆ str       ┆   ┆           ┆ bool      ┆           ┆        │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪════════╡
│ JWR_201915 ┆ Analysiere ┆ 1 Die rev ┆ EStG 1988 ┆ … ┆ ["EStG    ┆ false     ┆ 0.0       ┆ 0.0    │
│ 0007_20210 ┆ den        ┆ isionswer ┆ §4 Abs1;  ┆   ┆ 1988      ┆           ┆           ┆        │
│ 610J01     ┆ folgenden  ┆ bende    

In [ ]:
#Evaluate baseline model vs finetuned model 2nd round

if __name__ == "__main__":
    print(" Starting RoBERTa Before/After Evaluation...")
    
    # 1. Setup Paths (Update your dataset path if needed)
    DATASET_PATH = ["/mnt/red/red_hanka_bcthesis/llm/finetuning_KStGdataset_1.csv"]
    SSD_MODEL_PATH = "/mnt/windows/windows_hanka_bcthesis/llm/roberta_output/finetuned_roberta_austrian_law"
    
    BASELINE_CSV = "/mnt/red/red_hanka_bcthesis/llm/roberta_baseline_results_2.csv"
    FINETUNED_CSV = "/mnt/red/red_hanka_bcthesis/llm/roberta_finetuned_results_2.csv"
    
    # 2. Load the Full Data and Extract ALL Unique Citations
    # We must extract unique citations from the FULL dataset so the FAISS index knows all the laws.
    roberta = roberta()
    full_df = roberta.load_datasets(DATASET_PATH)
    
    unique_cits = (
        full_df.select(pl.col("parsed_citations").list.explode())
        .drop_nulls()
        .unique()
        .to_series()
        .to_list()
    )
    
    # 3. Cut the dataset to the first 200 for testing
    test_df = full_df.head(50)
    print(f"\nEvaluating on {test_df.height} cases. FAISS index contains {len(unique_cits)} citations.")

    # ==========================================
    # STAGE 1: BASELINE (Zero-Shot)
    # ==========================================
    print("\n" + "="*50)
    print("STAGE 1: BASELINE MODEL")
    print("="*50)
    base_model = roberta.load_model() 
    base_retriever = roberta_legal_retriever(model=base_model, unique_citations=unique_cits)
    # Call your new class method!
    base_retriever.generate_roberta_predictions(test_df, BASELINE_CSV, k=5)
    
    del base_model
    del base_retriever

    # ==========================================
    # STAGE 2: FINE-TUNED MODEL
    # ==========================================
    print("\n" + "="*50)
    print("STAGE 2: FINE-TUNED MODEL")
    print("="*50)
    # Load your trained model directly from the Windows SSD
    print(f"Loading trained weights from SSD: {SSD_MODEL_PATH}...")
    #trained_model = SentenceTransformer(SSD_MODEL_PATH)
    finetuned_model = roberta.load_model(SSD_MODEL_PATH)
    finetuned_model.max_seq_length = 512 # Re-apply VRAM fix
    
    finetuned_retriever = roberta_legal_retriever(model=finetuned_model, unique_citations=unique_cits)
    
    finetuned_retriever.generate_roberta_predictions(test_df, FINETUNED_CSV, k=5)
    
    # Free up VRAM
    del finetuned_model
    del finetuned_retriever

    # ==========================================
    # STAGE 3: RUN YOUR EVALUATOR
    # ==========================================
    print("\n" + "="*50)
    print("FINAL EVALUATION METRICS")
    print("="*50)
    
    # Instantiate your existing evaluator class
    eval_engine = evaluator(results_csv_path=BASELINE_CSV)  # You can initialize with any CSV, it just needs the path for internal use
    
    print("\n--- BASELINE SCORES ---")
    eval_engine.evaluate_results(
        output_path="/mnt/red/red_hanka_bcthesis/llm/eval_roberta_baseline_2.csv", 
        results_csv_path=BASELINE_CSV
    )
    
    print("\n--- FINE-TUNED SCORES ---")
    eval_engine.evaluate_results(
        output_path="/mnt/red/red_hanka_bcthesis/llm/eval_roberta_finetuned_2.csv", 
        results_csv_path=FINETUNED_CSV
    )

 Starting RoBERTa Before/After Evaluation...
Setup complete. Caching and Output mapped to SSD: {ssd_base_path}
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_KStGdataset_1.csv...
Successfully loaded and merged 1 datasets.
Total training cases available: 461

Evaluating on 50 cases. FAISS index contains 822 citations.

STAGE 1: BASELINE MODEL
Loading model: Stern5497/sbert-legal-xlm-roberta-base...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: Stern5497/sbert-legal-xlm-roberta-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== Building FAISS Index (Citation Strings Only) ===
Encoding 822 unique citations...


Batches:   0%|          | 0/26 [00:00<?, ?it/s]

...


=== Building FAISS Index (Citation Strings Only) ===
Encoding 822 unique citations...


Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Encoding took 0.30 seconds
Index built with 822 vectors.

Generating predictions -> /mnt/red/red_hanka_bcthesis/llm/roberta_finetuned_results_2.csv


100%|██████████| 50/50 [00:00<00:00, 69.67it/s]



FINAL EVALUATION METRICS

--- BASELINE SCORES ---
Loading results from /mnt/red/red_hanka_bcthesis/llm/roberta_baseline_results_2.csv...
shape: (50, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ instructi ┆ input     ┆ ground_tr ┆ … ┆ pred_list ┆ exact_mat ┆ precision ┆ recall   │
│ ---       ┆ on        ┆ ---       ┆ uth_label ┆   ┆ ---       ┆ ch        ┆ ---       ┆ ---      │
│ str       ┆ ---       ┆ str       ┆ ---       ┆   ┆ list[str] ┆ ---       ┆ f64       ┆ f64      │
│           ┆ str       ┆           ┆ str       ┆   ┆           ┆ bool      ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ JWR_20201 ┆ Analysier ┆ 1 Bei der ┆ 32006L011 ┆ … ┆ ["31969L0 ┆ false     ┆ 0.2       ┆ 0.076923 │
│ 50004_202 ┆ e den     ┆ Revisions ┆ 2 Mehrwer ┆   ┆ 335       ┆           ┆           ┆          │
│ 01207L01  ┆ folgenden ┆ werberin, ┆ 

In [1]:
import polars as pl
import csv
from tqdm import tqdm
from llms import roberta, roberta_legal_retriever

if __name__ == "__main__":
    print(" Starting finetuned RoBERTa-Only Inference for dataset_clean.csv...")
    
    # 1. SETUP DER PFADE
    ALL_DATASETS_PATHS = [
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_EStGdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_UStGdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_BAOdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_ASVGdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_DBAdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_GrEStGdataset_1.csv",
        "/mnt/red/red_hanka_bcthesis/llm/finetuning_KStGdataset_1.csv"
    ]
    SSD_MODEL_PATH = "/mnt/windows/windows_hanka_bcthesis/llm/roberta_output/finetuned_roberta_austrian_law"
    
    # Pfade für das Test-Set der Professorin
    PROJECT_INPUT_CSV = "/mnt/red/red_hanka_bcthesis/llm/dataset_clean.csv"
    PROJECT_OUTPUT_CSV = "/mnt/red/red_hanka_bcthesis/llm/model_output_roberta_finetuned_1.csv"
    
    # Der gewünschte Einleitungssatz
    PREFIX_PHRASE = "Nach sorgfältiger Prüfung Ihres Falles verweise ich auf die folgenden Paragraphen: "

    # 2. LADE ALLE DATENSÄTZE FÜR DEN FAISS-INDEX
    print("\n Loading training datasets to build complete FAISS index...")
    roberta_bot = roberta()
    full_df = roberta_bot.load_datasets(ALL_DATASETS_PATHS)
    
    unique_cits = (
        full_df.select(pl.col("parsed_citations").list.explode())
        .drop_nulls()
        .unique()
        .to_series()
        .to_list()
    )
    print(f"FAISS index will contain {len(unique_cits)} unique citations.")

    # 3. LADE DAS FINE-GETUNTE MODELL UND DEN RETRIEVER
    print(f"\n Loading trained weights from SSD: {SSD_MODEL_PATH}...")
    finetuned_model = roberta_bot.load_model(SSD_MODEL_PATH)
    finetuned_model.max_seq_length = 512 # VRAM Fix beibehalten
    
    retriever = roberta_legal_retriever(model=finetuned_model, unique_citations=unique_cits)

    # 4. VERARBEITE DIE 600 FÄLLE DER PROFESSORIN
    print(f"\n Loading project dataset from: {PROJECT_INPUT_CSV}")
    project_df = pl.read_csv(PROJECT_INPUT_CSV)
    
    print(f" Running Batch Inference on {project_df.height} questions (RoBERTa-Only Mode)...")
    
    with open(PROJECT_OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["id", "answer"]) # Exakt die geforderten Spalten
        
        for row in tqdm(project_df.iter_rows(named=True), total=project_df.height, desc="Querying RoBERTa"):
            q_id = row.get("id", "")
            query = row.get("prompt", "")
            
            # Hole die Top-5 Zitate von RoBERTa
            results = retriever.retrieve(query, k=5)
            
            # Extrahiere die Strings und baue die fertige Antwort zusammen
            citations = [r["citation"] for r in results]
            final_answer = PREFIX_PHRASE + "; ".join(citations) + "."
            
            # Schreibe in die CSV
            writer.writerow([q_id, final_answer])

    print(f"\n All done! The final RoBERTa-only answers are saved at: {PROJECT_OUTPUT_CSV}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
 Starting finetuned RoBERTa-Only Inference for dataset_clean.csv...

 Loading training datasets to build complete FAISS index...
Setup complete. Caching and Output mapped to SSD: {ssd_base_path}
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_EStGdataset_1.csv...
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_UStGdataset_1.csv...
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_BAOdataset_1.csv...
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_ASVGdataset_1.csv...
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_DBAdataset_1.csv...
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_GrEStGdataset_1.csv...
Reading /mnt/red/red_hanka_bcthesis/llm/finetuning_KStGdataset_1.csv...
Successfully loaded and merged 7 data

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== Building FAISS Index (Citation Strings Only) ===
Encoding 8485 unique citations...


Batches:   0%|          | 0/266 [00:00<?, ?it/s]

Encoding took 3.11 seconds
Index built with 8485 vectors.

 Loading project dataset from: /mnt/red/red_hanka_bcthesis/llm/dataset_clean.csv
 Running Batch Inference on 643 questions (RoBERTa-Only Mode)...


Querying RoBERTa: 100%|██████████| 643/643 [00:07<00:00, 81.19it/s]


 All done! The final RoBERTa-only answers are saved at: /mnt/red/red_hanka_bcthesis/llm/model_output_roberta_finetuned_1.csv
